In [ ]:

import numpy as np
import pandas as pd
import re
from scipy.interpolate import interp1d
from sklearn.neighbors import NearestNeighbors
import lightgbm as lgb


In [ ]:

DATASET_FILE = 'dataset(4).csv'
REFERENCE_FILE = 'finance2.csv'
OUTPUT_FILE = 'submission_final.csv'

EXPIRY_DT = pd.Timestamp('2026-01-27 15:30:00')

SEEDS = [17, 29, 53]

BLEND = 0.05
KNN_COUNT = 20

PATTERN = re.compile(
    r'NIFTY(\d+[A-Z]+\d{2})(\d+)(CE|PE)'
)


In [ ]:

def parse_name(name):

    match = PATTERN.match(name)

    return (
        int(match.group(2)),
        match.group(3)
    )


def interpolate_iv(x, y, target):

    model = interp1d(
        x,
        y,
        kind='linear',
        fill_value='extrapolate'
    )

    return float(model(target))


In [ ]:

df = pd.read_csv(DATASET_FILE)

reference = pd.read_csv(REFERENCE_FILE)

df['ts'] = pd.to_datetime(
    df['datetime'],
    format='%d-%m-%Y %H:%M'
)

df = df.sort_values('ts').reset_index(drop=True)

df['T'] = df['ts'].apply(
    lambda x: max(
        (EXPIRY_DT - x).total_seconds() / (365.25 * 24 * 3600),
        1e-6
    )
)

contracts = [
    c for c in df.columns
    if c.startswith('NIFTY')
]

strike_map = {
    c: parse_name(c)[0]
    for c in contracts
}

type_map = {
    c: parse_name(c)[1]
    for c in contracts
}


In [ ]:

iv = df[contracts].values

spot = df['underlying_price'].values

time_val = df['T'].values

N = len(df)

strike_arr = np.array([
    strike_map[c]
    for c in contracts
])

type_arr = np.array([
    1 if type_map[c] == 'CE' else 0
    for c in contracts
])


In [ ]:

def feature_vector(r, c):

    strike = strike_arr[c]

    option_type = type_arr[c]

    spot_price = spot[r]

    current_time = time_val[r]

    same_type = type_arr == option_type

    row_iv = iv[r][same_type]

    row_strike = strike_arr[same_type]

    valid = ~np.isnan(row_iv)

    row_iv = row_iv[valid]

    row_strike = row_strike[valid]

    if len(row_iv) >= 2:

        order = np.argsort(row_strike)

        row_iv = row_iv[order]

        row_strike = row_strike[order]

        pred = interpolate_iv(
            row_strike,
            row_iv,
            strike
        )

        avg = row_iv.mean()

        std = row_iv.std()

    else:

        pred = 0.15
        avg = 0.15
        std = 0.0

    prev_vals = iv[:r, c]

    prev_vals = prev_vals[
        ~np.isnan(prev_vals)
    ]

    if len(prev_vals) > 0:

        prev_iv = prev_vals[-1]

    else:

        prev_iv = avg

    return [
        strike,
        option_type,
        np.log(strike / spot_price),
        spot_price,
        current_time,
        avg,
        std,
        pred,
        prev_iv,
        strike - spot_price,
        abs(strike - spot_price)
    ]


In [ ]:

X_train = []

y_train = []

for r in range(N):

    for c in range(len(contracts)):

        value = iv[r, c]

        if not np.isnan(value):

            X_train.append(
                feature_vector(r, c)
            )

            y_train.append(value)

X_train = np.array(X_train)

y_train = np.array(y_train)


In [ ]:

X_test = []

test_ids = []

test_pos = []

for r in range(N):

    current_time = df.iloc[r]['datetime']

    for c, contract in enumerate(contracts):

        if np.isnan(iv[r, c]):

            X_test.append(
                feature_vector(r, c)
            )

            test_ids.append(
                f'{current_time}||{contract}'
            )

            test_pos.append((r, c))

X_test = np.array(X_test)


In [ ]:

predictions = np.zeros(len(X_test))

for seed in SEEDS:

    model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        reg_alpha=0.05,
        reg_lambda=0.05,
        random_state=seed,
        feature_fraction=0.85,
        bagging_fraction=0.85,
        bagging_freq=2,
        verbose=-1,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions += (
        model.predict(X_test) / len(SEEDS)
    )


In [ ]:

def knn_features(r, c):

    strike = strike_arr[c]

    return np.array([
        np.log(strike / spot[r]),
        r / N,
        abs(strike - spot[r]) / spot[r],
        float(type_arr[c])
    ])


observed_points = []
observed_values = []

for r in range(N):

    for c in range(len(contracts)):

        if not np.isnan(iv[r, c]):

            observed_points.append(
                knn_features(r, c)
            )

            observed_values.append(
                iv[r, c]
            )

observed_points = np.array(observed_points)

observed_values = np.array(observed_values)


In [ ]:

test_points = np.array([
    knn_features(r, c)
    for r, c in test_pos
])

nn = NearestNeighbors(
    n_neighbors=KNN_COUNT,
    algorithm='ball_tree'
)

nn.fit(observed_points)

distances, indices = nn.kneighbors(test_points)

weights = 1 / (distances + 1e-6)

weights = weights / weights.sum(
    axis=1,
    keepdims=True
)

knn_pred = (
    weights * observed_values[indices]
).sum(axis=1)


In [ ]:

final_pred = np.clip(
    (1 - BLEND) * predictions +
    BLEND * knn_pred,
    0.001,
    None
)

mapping = dict(
    zip(test_ids, final_pred)
)


In [ ]:

rows = []

for _, row in reference.iterrows():

    current_id = row['id']

    rows.append({
        'id': current_id,
        'value': mapping.get(
            current_id,
            row['value']
        )
    })

submission = pd.DataFrame(rows)

submission.to_csv(
    OUTPUT_FILE,
    index=False,
    float_format='%.16f'
)

print(submission.head())

print(submission.shape)
